# Fitting an orbit: circular, then Keplerian

This notebook fits an orbit to synthetic astrometry, building from a circular model up to a
full Keplerian one. As in the [abstract walkthrough](walkthrough), every step uses
photomancy's generic engine, here with an orbit forward model. The scene is a set of
orbital elements, and going from a circular to a Keplerian fit is just a change of which
PyTree leaves the partition opens up for fitting.


## The orbit as a scene

The fitted scene is an `Orbit` module of six elements. The period is parameterized as
`log P` rather than the semi-major axis directly, which keeps the elements at comparable
scales and the fit well-conditioned, the same reason photomancy's production orbit model
works in log-period. The forward model turns the elements into predicted sky positions
through orbix, and the likelihood is Gaussian in the measured offsets.


In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)

from orbix.equations import period_to_sma
from photomancy import LaplaceBackend, build_scene_logdensity
from photomancy.orbit.forward import predict_relative_astrometry
from photomancy.priors import Uniform

Ms = 1.989e30      # stellar mass [kg]
dist_pc = 10.0     # distance [pc]
P_true = 4000.0    # period [day]
err = 2e-3         # astrometric error [arcsec]
truth = dict(logP=float(np.log10(P_true)), e=0.15, inc=1.0, Omega=0.8, omega=1.2, M0=0.6)
times = jnp.linspace(0.0, P_true, 15)  # 15 epochs over one period


class Orbit(eqx.Module):
    # scaled Keplerian elements; log-period keeps the fit well-conditioned
    logP: jnp.ndarray
    e: jnp.ndarray
    inc: jnp.ndarray
    Omega: jnp.ndarray
    omega: jnp.ndarray
    M0: jnp.ndarray


def radec(o, ts):
    P = 10.0**o.logP
    a = period_to_sma(P, Ms)
    tp = -o.M0 * P / (2.0 * jnp.pi)
    return predict_relative_astrometry(
        ts, a, o.e, o.inc, o.Omega, jnp.cos(o.omega), jnp.sin(o.omega), tp, Ms, dist_pc
    )


true_orbit = Orbit(**{k: jnp.array(v) for k, v in truth.items()})
ra_t, dec_t = radec(true_orbit, times)
ra_obs = ra_t + err * jax.random.normal(jax.random.key(0), (len(times),))
dec_obs = dec_t + err * jax.random.normal(jax.random.key(1), (len(times),))
obs = jnp.concatenate([ra_obs, dec_obs])


def forward(o):
    r, d = radec(o, times)
    return jnp.concatenate([r, d])


def likelihood(pred):
    return -0.5 * jnp.sum(((pred - obs) / err) ** 2)


def fit_mask(scene, leaves_fn, n):
    m = jax.tree_util.tree_map(lambda _: False, scene)
    return eqx.tree_at(leaves_fn, m, [True] * n)


def plot_orbit(ax, o, **kw):
    r, d = radec(o, jnp.linspace(0.0, 10.0**o.logP, 400))
    ax.plot(np.array(r), np.array(d), **kw)


def sky_axes(title):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.scatter([0], [0], marker="*", c="orange", s=200, zorder=4, label="star")
    ax.scatter(np.array(ra_obs), np.array(dec_obs), c="crimson", s=30, zorder=3, label="astrometry")
    ax.set_aspect("equal")
    ax.invert_xaxis()
    ax.set_xlabel("RA offset [arcsec]")
    ax.set_ylabel("Dec offset [arcsec]")
    ax.set_title(title)
    return fig, ax


init = Orbit(
    logP=jnp.array(truth["logP"] + 0.02), e=jnp.array(0.2), inc=jnp.array(0.9),
    Omega=jnp.array(0.7), omega=jnp.array(1.0), M0=jnp.array(0.5),
)


## The data

Astrometry from a known, mildly eccentric orbit: 15 epochs over one period, each position
measured to 2 milliarcseconds. The star is at the origin and the planet traces the grey
ellipse.


In [ ]:
fig, ax = sky_axes("synthetic astrometry of a known orbit")
plot_orbit(ax, true_orbit, color="0.6", lw=1, label="true orbit")
ax.legend()
plt.show()


## 1. A circular first guess

A natural starting model is a circular orbit. Its period we take as a rough estimate from
the observation baseline; the remaining geometry, inclination, node, and phase, we fit by
opening just those three leaves of the partition. The best circular orbit captures the
rough size and orientation, but it cannot follow the faster motion near periapsis and the
slower motion near apoapsis, so it misses the data by about a tenth of an arcsecond, far
more than the measurement error.


In [ ]:
circ_scene = eqx.tree_at(
    lambda s: [s.logP, s.e, s.omega], init,
    [jnp.array(truth["logP"]), jnp.array(0.0), jnp.array(0.0)],  # period estimate, e = 0
)
circ_prior = Uniform(low=jnp.array([0.0, 0.0, -jnp.pi]), high=jnp.array([jnp.pi, 2 * jnp.pi, jnp.pi]))
circ_mask = fit_mask(circ_scene, lambda s: [s.inc, s.Omega, s.M0], 3)

ld_c, z0_c, unravel_c = build_scene_logdensity(circ_scene, forward, likelihood, circ_prior, filter_spec=circ_mask)
post_c = LaplaceBackend(n_steps=600, min_eigenvalue=1e-8).run(ld_c, z0_c)
orbit_c = eqx.combine(unravel_c(post_c.mean), eqx.partition(circ_scene, circ_mask)[1])
chi2_c = float(jnp.sum(((forward(orbit_c) - obs) / err) ** 2))
print(f"circular chi^2 = {chi2_c:.0f}  (with {2 * len(times)} measurements)")

fig, ax = sky_axes("circular fit: right size, wrong shape")
plot_orbit(ax, orbit_c, color="gold", lw=1.5, ls="--", label="circular fit")
ax.legend()
plt.show()


## 2. The full Keplerian fit

Relaxing the eccentricity and refining the period gives the full six-element fit, the same
`build_scene_logdensity` call with two more leaves in the partition. The Laplace fit
recovers the period, eccentricity, and inclination, and matches the astrometry to within
the noise. Astrometry alone leaves a degeneracy between the node and the argument of
periapsis, so those two are not individually pinned down even though the orbit they trace
is; a radial-velocity measurement or a longer arc would break it.


In [ ]:
kep_prior = Uniform(
    low=jnp.array([3.0, 0.0, 0.0, 0.0, 0.0, -jnp.pi]),
    high=jnp.array([4.2, 0.9, jnp.pi, 2 * jnp.pi, 2 * jnp.pi, jnp.pi]),
)
kep_mask = fit_mask(init, lambda s: [s.logP, s.e, s.inc, s.Omega, s.omega, s.M0], 6)

ld_k, z0_k, unravel_k = build_scene_logdensity(init, forward, likelihood, kep_prior, filter_spec=kep_mask)
post_k = LaplaceBackend(n_steps=600, min_eigenvalue=1e-8).run(ld_k, z0_k)
orbit_k = eqx.combine(unravel_k(post_k.mean), eqx.partition(init, kep_mask)[1])
chi2_k = float(jnp.sum(((forward(orbit_k) - obs) / err) ** 2))

print(f"recovered: logP={float(orbit_k.logP):.3f}  e={float(orbit_k.e):.3f}  inc={float(orbit_k.inc):.3f}")
print(f"truth:     logP={truth['logP']:.3f}  e={truth['e']:.3f}  inc={truth['inc']:.3f}")
print(f"chi^2 = {chi2_k:.1f}  (circular was {chi2_c:.0f})")

fig, ax = sky_axes("Keplerian fit matches the data")
plot_orbit(ax, orbit_c, color="gold", lw=1.2, ls="--", label="circular")
plot_orbit(ax, orbit_k, color="cyan", lw=1.8, label="Keplerian")
ax.legend()
plt.show()


## 3. Does the data need eccentricity?

The Laplace evidence answers this directly. The ratio of the Keplerian and circular
evidences is a Bayes factor, and here it is overwhelmingly in favor of the eccentric model:
the circular residuals are tens of sigma, so the two extra parameters are decisively
warranted. Had the true orbit been nearly circular, the evidence would have preferred the
simpler model through the Occam penalty the marginal likelihood carries.


In [ ]:
print(f"Keplerian log Z = {float(post_k.evidence):.1f}")
print(f"circular  log Z = {float(post_c.evidence):.1f}")
print(f"log Bayes factor (Keplerian vs circular) = {float(post_k.evidence - post_c.evidence):.1f}")


## What this shows, and what comes next

Going from a circular to a Keplerian fit was a change of partition: the same scene, forward
model, and engine, with two more leaves opened for fitting, and the evidence then judged
whether those leaves earned their place.

This was a guided fit with a rough period in hand. Fitting an orbit blind, where the period
itself is unknown, is genuinely multimodal, since many periods alias to similar tracks over
a short arc. photomancy's orbit module handles that with Thiele-Innes and grid-search
initializers and an evidence-weighted mixture, and the [abstract walkthrough](walkthrough)
shows how expected information gain then chooses the most informative next epoch.
